# KG population demo from processed reliability documents

This notebook picks up **after** your Stage 1–6 parsing workflow and uses the workflow-oriented Neo4j classes to:

1. load processed document outputs,
2. derive lightweight `document` records from the processed text records,
3. build a graph batch with `build_graph_from_workflow_artifacts`,
4. preview the nodes and edges,
5. optionally ingest the graph into Neo4j.

It is designed to be a companion to `stage1_6_existing_methods_demo.ipynb`.


In [1]:
from pathlib import Path
print(Path().resolve())
print(list(Path().resolve().parents))

/Users/mandd/projects/DACKAR/src/dackar/RCA/demos
[PosixPath('/Users/mandd/projects/DACKAR/src/dackar/RCA'), PosixPath('/Users/mandd/projects/DACKAR/src/dackar'), PosixPath('/Users/mandd/projects/DACKAR/src'), PosixPath('/Users/mandd/projects/DACKAR'), PosixPath('/Users/mandd/projects'), PosixPath('/Users/mandd'), PosixPath('/Users'), PosixPath('/')]


In [2]:
from __future__ import annotations

import sys
import os
import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import pandas as pd

from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import pandas as pd

rca_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(rca_root)

dackar_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(dackar_root)

from kg.kg_schema_builder_workflow import (
    apply_schema_constraints,
    build_graph_from_workflow_artifacts,
    load_and_merge_schemas,
)
from kg.py2neo_workflow import Py2Neo
from kg_population_helpers import load_processed_records_from_output


In [3]:
# ----- configuration -----
# Point this to the output directory created by the Stage 1-6 document parsing notebook.
OUTPUT_ROOT = Path("./1-6pipeline_demo_existing_methods")

# The stage notebook stores enriched JSONL files under the output root. If your layout differs,
# update ENRICHED_GLOBS below.
ENRICHED_GLOBS = [
    "**/*enriched*.jsonl",
    "**/*processed*.jsonl",
]

SCHEMA_PATHS = [
    "../../knowledge_graph/schemas/customMbseSchema.toml",
    "../../knowledge_graph/schemas/documentSchema.toml",
    "../../knowledge_graph/schemas/conditionReportSchema.toml",
    "../../knowledge_graph/schemas/workOrderSchema.toml",
    "../../knowledge_graph/schemas/causalSchema.toml",
    "../../knowledge_graph/schemas/fmeaSchema.toml",
]

# Neo4j connection: leave INGEST_TO_NEO4J = False for a dry run.
INGEST_TO_NEO4J = True
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "123456789")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE") or None
RESET_DATABASE_FIRST = False
CREATE_CONSTRAINTS = True


In [4]:
documents, processed_text_records, enriched_files = load_processed_records_from_output(OUTPUT_ROOT, ENRICHED_GLOBS)

print(f"Found {len(enriched_files)} enriched JSONL files")
print(f"Derived {len(documents)} document records")
print(f"Loaded {len(processed_text_records)} processed text records")

display(pd.DataFrame({"enriched_jsonl": [str(p) for p in enriched_files]}))


Found 1 enriched JSONL files
Derived 1 document records
Loaded 9 processed text records


,enriched_jsonl
0,1-6pipeline_demo_existing_methods/example_CR_2...


In [5]:
# Preview the derived document objects that will be passed into the workflow KG builder.
pd.DataFrame(documents).fillna("")


,doc_id,doc_type,title,equipment_ids,component_refs,authority_level,source_file
0,ad61d26ecf6e,CR,ad61d26ecf6e_chunks_enriched.jsonl,"[CA-1, CA-2, CA-3, FT-1102, P-101A, PT-1102, V...","[{'component_id': 'FT-1102'}, {'component_id':...",informational,1-6pipeline_demo_existing_methods/example_CR_2...


In [6]:
# Preview a sample processed text record.
sample_record = processed_text_records[0]
sample_record


{'record_id': 'ad61d26ecf6e::0',
 'doc_id': 'ad61d26ecf6e',
 'doc_type': 'CR',
 'chunk_index': 0,
 'embedding_text': "SCOPE: | Field | Value | Field | Value | |----------|-----------------------------|----------------|--------------------| | CR ID | CR-2026-00123 | Date Initiated | 2026-02-06 | | System | AFW | Equipment | P-101A | | Location | Turbine Building Elev. 468' | Initiator | Operator (Example) |\n\nAFW Train B remained available. No safety injection or reactor trip occurred. This CR is categorized as low safety\nSYSTEMS: \nEQUIPMENT: P-101A, VB-101A, FT-1102, PT-1102\nCOMPONENTS: P-101A, VB-101A, FT-1102, PT-1102\nSYMPTOMS/OUTCOMES: \nMECHANISMS: \nDIAGNOSTICS: test\nACTIONS: \nNUMBERS/LIMITS: 101a, 101a, 5.0 mils\nKEYWORDS: field, value, field value, ----------, -----------------------------, ----------------, --------------------, cr-2026-00123, date, initiated, 2026-02-06, afw, safety, train, remained, available, injection, reactor, trip, occurred, categorized, low, signi

In [7]:
schema = load_and_merge_schemas(SCHEMA_PATHS)

nodes, edges = build_graph_from_workflow_artifacts(
    SCHEMA_PATHS,
    documents=documents,
    processed_text_records=processed_text_records,
)

print(f"Graph batch built with {len(nodes)} nodes and {len(edges)} edges")


Graph batch built with 21 nodes and 43 edges


In [8]:
nodes_df = pd.DataFrame([
    {
        "id": n["id"],
        "label": n["label"],
        **{k: v for k, v in n["attrs"].items() if k in {"id", "doc_key", "doc_type", "record_key", "source_key", "kind", "label"}},
    }
    for n in nodes
])

nodes_df = pd.DataFrame(nodes) if nodes else pd.DataFrame(columns=["id", "label", "properties"])

normalized_edges = []
for e in edges or []:
    normalized_edges.append({
        "source": e.get("source") or e.get("from"),
        "target": e.get("target") or e.get("to"),
        "type": e.get("type") or e.get("relation") or e.get("edge_type"),
        "properties": e.get("properties", {}),
    })

edges_df = pd.DataFrame(normalized_edges, columns=["source", "target", "type", "properties"])

print("Node labels")
display(nodes_df["label"].value_counts().rename_axis("label").reset_index(name="count"))

print("Edge types")
display(edges_df["type"].value_counts().rename_axis("type").reset_index(name="count"))


Node labels


,label,count
0,mbse_entity,11
1,ProcessedTextRecord,9
2,condition_report,1


Edge types


,type,count
0,references_entity,27
1,derived_from_document,9
2,mentions,7


In [9]:
display(nodes_df.head(25))
display(edges_df.head(25))


,id,label,attrs
0,DOC:ad61d26ecf6e,condition_report,"{'doc_id': 'ad61d26ecf6e', 'doc_type': 'CR', '..."
1,ASSET:CA-1,mbse_entity,"{'source_key': 'CA-1', 'kind': 'asset', 'id': ..."
2,ASSET:CA-2,mbse_entity,"{'source_key': 'CA-2', 'kind': 'asset', 'id': ..."
3,ASSET:CA-3,mbse_entity,"{'source_key': 'CA-3', 'kind': 'asset', 'id': ..."
4,ASSET:FT-1102,mbse_entity,"{'source_key': 'FT-1102', 'kind': 'asset', 'id..."
5,ASSET:P-101A,mbse_entity,"{'source_key': 'P-101A', 'kind': 'asset', 'id'..."
6,ASSET:PT-1102,mbse_entity,"{'source_key': 'PT-1102', 'kind': 'asset', 'id..."
7,ASSET:VB-101A,mbse_entity,"{'source_key': 'VB-101A', 'kind': 'asset', 'id..."
8,ad61d26ecf6e::0,ProcessedTextRecord,"{'record_key': 'ad61d26ecf6e::0', 'doc_id': 'a..."
9,CMP:P-101A,mbse_entity,"{'source_key': 'P-101A', 'kind': 'component', ..."


,source,target,type,properties
0,DOC:ad61d26ecf6e,ASSET:CA-1,mentions,{}
1,DOC:ad61d26ecf6e,ASSET:CA-2,mentions,{}
2,DOC:ad61d26ecf6e,ASSET:CA-3,mentions,{}
3,DOC:ad61d26ecf6e,ASSET:FT-1102,mentions,{}
4,DOC:ad61d26ecf6e,ASSET:P-101A,mentions,{}
5,DOC:ad61d26ecf6e,ASSET:PT-1102,mentions,{}
6,DOC:ad61d26ecf6e,ASSET:VB-101A,mentions,{}
7,ad61d26ecf6e::0,DOC:ad61d26ecf6e,derived_from_document,{}
8,ad61d26ecf6e::0,ASSET:P-101A,references_entity,{}
9,ad61d26ecf6e::0,ASSET:VB-101A,references_entity,{}


In [10]:
# Optional: export the graph batch so it can be shown without Neo4j.
EXPORT_DIR = OUTPUT_ROOT / "kg_population_demo"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

with (EXPORT_DIR / "nodes.json").open("w", encoding="utf-8") as f:
    json.dump(nodes, f, indent=2, ensure_ascii=False)
with (EXPORT_DIR / "edges.json").open("w", encoding="utf-8") as f:
    json.dump(edges, f, indent=2, ensure_ascii=False)

print("Exported graph batch to", EXPORT_DIR)


Exported graph batch to 1-6pipeline_demo_existing_methods/kg_population_demo


In [11]:
# Optional Neo4j ingestion.
# Set INGEST_TO_NEO4J = True in the configuration cell to enable this block.
if INGEST_TO_NEO4J:
    client = Py2Neo(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
    try:
        if RESET_DATABASE_FIRST:
            client.reset(db=NEO4J_DATABASE)
        if CREATE_CONSTRAINTS:
            apply_schema_constraints(client, SCHEMA_PATHS, database=NEO4J_DATABASE)
        client.upsert_nodes_batch(nodes, db=NEO4J_DATABASE)
        client.upsert_edges_batch(edges, db=NEO4J_DATABASE)
        print(f"Ingested {len(nodes)} nodes and {len(edges)} edges into Neo4j")
    finally:
        client.close()
else:
    print("Dry run only. Set INGEST_TO_NEO4J = True to write to Neo4j.")


Ingested 21 nodes and 43 edges into Neo4j


In [12]:
# Optional verification query after ingestion.
if INGEST_TO_NEO4J:
    client = Py2Neo(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
    try:
        counts = client.query(
            "MATCH (n) RETURN labels(n) AS labels, count(*) AS count ORDER BY count DESC",
            db=NEO4J_DATABASE,
        )
        display(pd.DataFrame([dict(r) for r in counts]))
    finally:
        client.close()


,labels,count
0,[mbse_entity],11
1,[ProcessedTextRecord],9
2,[condition_report],1
